In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

REQUIRED_FILES = [
    "world-happiness-report-2021.csv",
    "world-happiness-report.csv",
]

def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

# GitHub Auto-Download
if running_in_colab():
    GITHUB_REPO_URL = "https://github.com/ruonezhang/world-happiness-report-powerbi-analysis.git"
    REPO_NAME = "world-happiness-report-powerbi-analysis"

    if not Path(REPO_NAME).exists():
        print("Colab environment detected. Cloning data from GitHub...")
        os.system(f"git clone {GITHUB_REPO_URL}")
        os.chdir(REPO_NAME)
        print("Data successfully downloaded and environment set up.")
# ==================================================================

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "data",
    Path("/content"),
    Path("/content") / "data",
]

def find_data_file(filename):
    for data_dir in candidate_dirs:
        path = data_dir / filename
        if path.exists():
            return path
    return None

# Locate the files
paths = {filename: find_data_file(filename) for filename in REQUIRED_FILES}
missing_files = [filename for filename, path in paths.items() if path is None]

# If files are missing, raise a clean English error
if missing_files:
    raise FileNotFoundError(
        "Missing required files: " + ", ".join(missing_files) + ". "
        "Please ensure they are in the current directory or a data/ folder."
    )

path_2021 = paths["world-happiness-report-2021.csv"]
path_panel = paths["world-happiness-report.csv"]

print(f"Data files successfully loaded:\n- 2021 Data: {path_2021}\n- Historical Data: {path_panel}")

Colab environment detected. Cloning data from GitHub...
Data successfully downloaded and environment set up.


FileNotFoundError: Missing required files: world-happiness-report-2021.csv, world-happiness-report.csv. Please ensure they are in the current directory or a data/ folder.

In [ ]:
import pandas as pd
df_2021 = pd.read_csv(path_2021)
df_panel = pd.read_csv(path_panel)


columns_to_drop = [
    "Standard error of ladder score",
    "upperwhisker",
    "lowerwhisker",
    "Ladder score in Dystopia",
    "Explained by: Log GDP per capita",
    "Explained by: Generosity",
    "Explained by: Social support",
    "Explained by: Healthy life expectancy",
    "Explained by: Freedom to make life choices",
    "Explained by: Perceptions of corruption",
    "Dystopia + residual",
]

df_2021 = df_2021.drop(columns=columns_to_drop, errors="ignore")

df_2021 = df_2021.rename(columns={
    "Logged GDP per capita": "GDP per capita",
    "Freedom to make life choices": "Life choices freedom",
})

df_panel = df_panel.rename(columns={
    "Life Ladder": "Ladder score",
    "Log GDP per capita": "GDP per capita",
    "Healthy life expectancy at birth": "Healthy life expectancy",
    "Freedom to make life choices": "Life choices freedom",
})

df_panel = df_panel.dropna(subset=["Ladder score", "year"])

display(df_2021.head(2))
display(df_panel.head(2))


**Obeservation**:\
Detected 3 columns missing from **`df_2021`** ( `Year`, `Positive affect`, `Negative affect` ) and 1 column missing from **`df_panel`**( `Regional indicator` ). In order to be able to concat these 2 tables into 1 fact table **`fact_happiness`** to facilitate the data modeling in PowerBI, we will add up missing columns and align these 2 tables in terms of column order.


In [ ]:
df_2021["Year"] = 2021
df_panel = df_panel.rename(columns= {"year":"Year"})

region_map = df_2021.set_index("Country name")["Regional indicator"]

df_panel["Regional indicator"] = df_panel["Country name"].map(region_map)

df_2021["Positive affect"] = pd.NA
df_2021["Negative affect"] = pd.NA

cols = [
    "Country name", "Year", "Regional indicator",
    "Ladder score", "GDP per capita", "Social support",
    "Healthy life expectancy", "Life choices freedom",
    "Generosity", "Perceptions of corruption",
    "Positive affect", "Negative affect"
]

fact_happiness = pd.concat([df_panel[cols],df_2021[cols]], axis = 0, ignore_index=True)

fact_happiness["Year"].head()




Check if country names are alligned and if "country + year“
 key is unique

In [ ]:
fact_happiness.duplicated(["Country name", "Year"]).sum()

In [ ]:
country_2021 = set(df_2021["Country name"])
country_panel = set(df_panel["Country name"])
print("Only in 2021:")
print(country_2021 - country_panel)

print("\nOnly in panel:")
print(country_panel - country_2021)

print(len(country_2021))
print(len(country_panel))

**Observation**:\
A country consistency check was performed between the 2021 and panel datasets. The panel dataset contained additional countries that were not present in the 2021 dataset. These records were retained because they represent valid observations rather than naming inconsistencies.

However, for countries that only exist in **`df_panel`**, `Regional indicator` were not added by map() as they don't exist in **`df_2021`**. As a result, we will manually add regional indicators for these countries.



In [ ]:
df_2021["Regional indicator"].unique()

In [ ]:
additional_region_map= {"Somaliland region": "Sub-Saharan Africa",
    "South Sudan": "Sub-Saharan Africa",
    "Guyana": "Latin America and Caribbean",
    "Central African Republic": "Sub-Saharan Africa",
    "Congo (Kinshasa)": "Sub-Saharan Africa",
    "Bhutan": "South Asia",
    "Djibouti": "Middle East and North Africa",
    "Qatar": "Middle East and North Africa",
    "Belize": "Latin America and Caribbean",
    "Sudan": "Sub-Saharan Africa",
    "Suriname": "Latin America and Caribbean",
    "Cuba": "Latin America and Caribbean",
    "Syria": "Middle East and North Africa",
    "Somalia": "Sub-Saharan Africa",
    "Oman": "Middle East and North Africa",
    "Trinidad and Tobago": "Latin America and Caribbean",
    "Angola": "Sub-Saharan Africa"}
fact_happiness.loc[fact_happiness["Regional indicator"].isna(),'Regional indicator'] = (
    fact_happiness.loc[fact_happiness["Regional indicator"].isna(),"Country name"]
    .map(additional_region_map))

In [ ]:
print(fact_happiness["Regional indicator"].isna().sum())

Generate the first dimension table **`dim_country`**


In [ ]:
dim_country = fact_happiness[["Country name", "Regional indicator"]].drop_duplicates().reset_index(drop = True)
dim_country.head()

Output **`fact_happiness`** and **`dim_country`** as csv.document. Will use **DAX** to create "Date table" and "Measures table"  

In [ ]:
fact_happiness.to_csv("fact_happiness.csv", index=False)

dim_country.to_csv("dim_country.csv", index=False)